<a href="https://colab.research.google.com/github/samnocera05-ui/mis433/blob/main/In_class_vibecoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
import requests
from google.colab import ai
from IPython.display import Image, display

In [56]:
breeds = requests.get("https://dog.ceo/api/breeds/list/all").json()["message"]
print("Main breeds:", len(breeds))
print("Selectable breeds:", sum(len(s) if s else 1 for s in breeds.values()))

Main breeds: 108
Selectable breeds: 165


In [57]:
# Option A: type a specific breed (leave blank to use preferences below)
breed_input = ""  #@param {type:"string"}

# Option B: adopter preferences
size = "Large"  #@param ["Any", "Small", "Medium", "Large"]
fur_color = "White"  #@param ["Any", "Black", "White", "Brown", "Golden", "Gray", "Spotted/Mixed"]
energy = "Medium"  #@param ["Low", "Medium", "High"]
home = "Apartment"  #@param ["Apartment", "House with yard"]

budget = "Over $500"  #@param ["Up to $100", "Up to $250", "Up to $500", "Over $500"]

all_breeds = requests.get("https://dog.ceo/api/breeds/list/all").json()["message"]
paths = [f"{b}/{s}" if s else b for b, subs in all_breeds.items() for s in (subs or [None])]

if breed_input.strip():
    # Typed breed: "golden retriever" -> "retriever/golden"
    words = breed_input.lower().strip().split()
    pick = f"{words[-1]}/{' '.join(words[:-1])}" if len(words) > 1 else words[0]
    reason_prompt = "In 2 sentences, describe the {breed} dog breed: size, temperament, and energy level."
else:
    # AI picks the best match from the real breed list
    pick = ai.generate_text(f"""From this list of dog breeds: {', '.join(paths)}
Pick the ONE breed that best fits: size {size}, fur color {fur_color}, energy level {energy}, living in a {home}.
Return only the breed exactly as written in the list, nothing else.""").strip().strip("`.").lower()
    reason_prompt = (f"In 2 sentences, explain why a {{breed}} fits someone wanting a {size.lower()}, "
                     f"{fur_color.lower()}-coated, {energy.lower()}-energy dog living in a {home.lower()}.")

if pick in paths:
    img_url = requests.get(f"https://dog.ceo/api/breed/{pick}/images/random").json()["message"]
    breed = " ".join(reversed(pick.split("/"))).title()
    print("Your match:", breed)
    display(Image(url=img_url, width=400))
    print("\nAbout this breed:\n", ai.generate_text(reason_prompt.format(breed=breed)))
else:
    breed = None
    name = breed_input.title() if breed_input.strip() else "dogs matching your preferences"
    print(f"Sorry, there are no {name} available for adoption right now.")

Your match: Clumber



About this breed:
 As the largest of the spaniel breeds, the Clumber provides the substantial presence of a big dog naturally paired with a predominantly white, beautiful coat. Their dignified, medium-energy temperament and notoriously quiet nature make them gentle "couch potatoes" indoors who easily adapt to apartment living with just a couple of moderate daily walks.
Your match: Pyrenees



About this breed:
 The Great Pyrenees perfectly matches your visual preferences as a majestic, large breed renowned for its lush, thick white coat. Despite their imposing size, their remarkably calm indoor demeanor and moderate exercise needs make them peaceful "couch potatoes" that can thrive in an apartment with just a few dedicated daily walks.


In [58]:
from IPython.display import Markdown, HTML

if not breed:
    raise SystemExit("No dog available, so no profile was generated.")

def ask(prompt):
    return ai.generate_text(prompt).replace("```markdown", "").replace("```", "").strip()

context = f"Adoptable dog, breed: {breed}."

# Output 1: adoption profile
profile = ask(f"""{context} Write a shelter adoption profile in Markdown using exactly this format:
## 🐾 Meet [made-up name]!
**Breed:** {breed}
**Age:** [age]
**Personality:** [3 traits, comma separated]

[2 warm sentences about the dog]

**Ideal home:** [one sentence]
Return only the Markdown.""")

# Output 2: adopter's email to the shelter
email = ask(f"""{context} The adopter wants a {size.lower()}, {fur_color.lower()}-coated, {energy.lower()}-energy dog
for a {home.lower()}. Their adoption fee budget is {budget.lower()}.
Write a short, friendly email from the adopter to the shelter asking to meet this dog.
Politely mention the budget and ask what the adoption fee includes (e.g., vaccines, microchip, spay/neuter).
Format: 'Subject: ...' on the first line, then 5 sentences. Return plain text only.""")

# Split subject from body
lines = email.split("\n", 1)
subject = lines[0].replace("Subject:", "").strip()
body = lines[1].strip() if len(lines) > 1 else ""

# Display profile
display(Markdown(profile))

# Display email with dog photo + mock Send button
display(HTML(f"""
<div style="background:#fff;color:#222;border:1px solid #ddd;border-radius:12px;max-width:560px;font-family:sans-serif;margin-top:16px;overflow:hidden">
  <div style="background:#f1f3f4;padding:12px 16px;border-bottom:1px solid #ddd">
    <div><b>To:</b> adoptions@shelter.org</div>
    <div><b>Subject:</b> {subject}</div>
  </div>
  <div style="padding:16px">
    <div style="white-space:pre-wrap;line-height:1.5">{body}</div>
    <img src="{img_url}" style="width:100%;border-radius:8px;margin-top:14px">
    <div style="font-size:12px;color:#666;margin-top:4px">📎 Photo of the {breed} I'm interested in</div>
    <button
      onclick="this.innerText='✅ Email sent to shelter!'; this.style.background='#188038'; this.disabled=true;"
      style="margin-top:16px;background:#1a73e8;color:#fff;border:none;border-radius:20px;padding:10px 24px;font-size:14px;cursor:pointer">
      📤 Send
    </button>
  </div>
</div>
"""))

# Plain-text version too
print("\n--- Plain text email ---\n")
print(email)

## 🐾 Meet Barnaby!
**Breed:** Pyrenees
**Age:** 3 years old
**Personality:** Gentle, watchful, affectionate

Barnaby is a gentle giant who loves leaning against your legs for quiet cuddles and keeping a calm, watchful eye over his favorite people. With his majestic cloud-like coat and soulful brown eyes, he brings a soothing, peaceful presence to every room he enters.

**Ideal home:** A loving family with a securely fenced yard where he can lounge in the cool breeze and be showered with affection.


--- Plain text email ---

Subject: Inquiry About Meeting Your Adoptable Pyrenees

Hello, I recently saw the adoptable Pyrenees on your website and would love the opportunity to schedule a time to meet him. I live in an apartment and have been searching specifically for a large, white-coated, medium-energy companion who would thrive in a calm, loving environment. Regarding costs, my adoption fee budget is comfortably over $500 to ensure a smooth transition. Could you please share what the adoption fee includes, such as current vaccines, a microchip, and spay or neuter services? Thank you so much for your dedication to these animals, and I look forward to hearing from you soon!
